# Week 6 · Session 2 — Busan Air Quality Station Pipeline
### Introduction to Data Science

**NAME**: <br>Hyeseong Park 박혜성
**STUDENT ID**: 201524471

**How to use this notebook:**
Follow the slide deck alongside this notebook. Each stage header matches a slide.
Cells marked with `# YOUR CODE HERE` or `___` are yours to complete.

---
| | |
|---|---|
| **Dataset** | `busan_aq_stations.csv` — 120 monitoring stations across Busan's 16 districts |
| **Reference** | McKinney, *Python for Data Analysis* 3rd Ed. — Chapter 5 §5.1–5.2 |
| **Pipeline** | Load → Inspect → Index → Filter → Compute → Deduplicate |
---

## Setup — Run Before Anything Else

In [60]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy: ", np.__version__)

pandas: 3.0.2
numpy:  2.4.4


> **Note:** The `busan_aq_stations.csv` is in PLATO post.
> If you need to generate it yourself, run the generator cell in the Appendix at the bottom.

---
## Stage 1 — Load & Inspect

**Goal:** Load the CSV and run the four standard inspection commands.
Always do this before writing a single transformation.
Rule: *"know what you have before you touch it."*

### 1.1 — Load the file and check shape + dtypes

In [61]:
# Load busan_aq_stations.csv into a DataFrame called df
# YOUR CODE HERE
df = pd.read_csv("busan_aq_stations.csv")

# Print shape (rows, columns)
print("Shape:", df.shape)

# Print the dtype of every column
print()
print("Dtypes:")
print(df.dtypes)

Shape: (120, 10)

Dtypes:
station_id                   str
district                     str
latitude                 float64
longitude                float64
station_type                 str
install_year               int64
pm25_annual_mean         float64
no2_annual_mean          float64
is_active                   bool
last_calibration_date        str
dtype: object


*After running — answer this:*
- Which column has `dtype=object` that should be a date?
- Which numeric columns are `float64`? Why does that matter for NaN handling?

### 1.2 — First five rows

In [62]:
# Show the first 5 rows of the DataFrame
df.head()

,station_id,district,latitude,longitude,station_type,install_year,pm25_annual_mean,no2_annual_mean,is_active,last_calibration_date
0,BUS-001,Nam-gu,35.32548,128.99372,background,2011,44.4,30.4,True,2023-03-15
1,BUS-002,Seo-gu,35.25493,129.17816,background,2022,28.6,32.7,True,2023-11-30
2,BUS-003,Seo-gu,35.30958,129.29169,background,2013,48.4,13.0,True,2023-03-15
3,BUS-004,Geumjeong-gu,35.21178,129.13685,roadside,2020,40.8,24.6,True,2023-01-20
4,BUS-005,Nam-gu,35.24785,128.90289,roadside,2006,12.4,11.5,True,2023-08-10


### 1.3 — info() — the fastest null-count command

In [63]:
# Run df.info() — shows non-null counts, dtypes, and memory usage in one call
# Any column with non-null count < 120 has missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   station_id             120 non-null    str    
 1   district               120 non-null    str    
 2   latitude               120 non-null    float64
 3   longitude              120 non-null    float64
 4   station_type           120 non-null    str    
 5   install_year           120 non-null    int64  
 6   pm25_annual_mean       110 non-null    float64
 7   no2_annual_mean        111 non-null    float64
 8   is_active              120 non-null    bool   
 9   last_calibration_date  120 non-null    str    
dtypes: bool(1), float64(4), int64(1), str(4)
memory usage: 8.7 KB


### 1.4 — Explicit null count per column

In [64]:
# Count null values in each column
print("Null counts:")
print(df.isnull().sum())

# Overall null rate
print()
print(f"Overall null rate: {df.isnull().sum().sum() / df.size:.1%}")

Null counts:
station_id                0
district                  0
latitude                  0
longitude                 0
station_type              0
install_year              0
pm25_annual_mean         10
no2_annual_mean           9
is_active                 0
last_calibration_date     0
dtype: int64

Overall null rate: 1.6%


### 1.5 — describe() for numeric summary

In [65]:
# Numeric summary statistics — count, mean, std, min, quartiles, max
# Tip: compare 'count' to 120. A lower count means NaN present.
df.describe()

,latitude,longitude,install_year,pm25_annual_mean,no2_annual_mean
count,120.000000,120.000000,120.000000,110.000000,111.000000
mean,35.210449,129.082270,2009.950000,33.202727,27.374775
std,0.109347,0.116878,7.450509,17.354911,10.463221
min,35.021950,128.884510,1998.000000,8.300000,10.500000
25%,35.102627,128.981205,2004.000000,21.300000,18.050000
50%,35.234790,129.082515,2009.000000,30.250000,27.500000
75%,35.302865,129.174143,2017.000000,43.675000,36.550000
max,35.379680,129.298100,2022.000000,119.700000,44.700000


### 1.6 — value_counts() for categorical columns

In [66]:
# How many stations of each type?
print("Station types:")
print(df["station_type"].value_counts())

print()
# How many unique districts?
print("Unique districts:", df["district"].nunique())

print()
# Active vs inactive breakdown
print("Active status:")
print(df["is_active"].value_counts())

Station types:
station_type
roadside      48
background    44
industrial    28
Name: count, dtype: int64

Unique districts: 16

Active status:
is_active
True     102
False     18
Name: count, dtype: int64


**Stage 1 checkpoint** — Before continuing, confirm all of these:
- `df.shape` == `(120, 9)`
- `pm25_annual_mean` and `no2_annual_mean` have ~8–10 NaN each
- Three station types are present: roadside, background, industrial
- `is_active` has both `True` and `False` values

---
## Stage 2 — Set the Index

**Goal:** Promote `station_id` to the row index so every subsequent operation
uses meaningful labels instead of integers 0–119.
Then understand the critical difference between `.index=` and `.reindex()`.

### 2.1 — Set station_id as the index on load

In [67]:
# Re-load the CSV and promote station_id to the row index in one step
# Hint: use the index_col parameter of pd.read_csv()

df = pd.read_csv("busan_aq_stations.csv", index_col="station_id")

# Verify: index name should be 'station_id'
print("Index name:   ", df.index.name)
print("First 5 IDs:  ", df.index[:5].tolist())
print("Is unique?    ", df.index.is_unique)          # will be False — duplicates exist

Index name:    station_id
First 5 IDs:   ['BUS-001', 'BUS-002', 'BUS-003', 'BUS-004', 'BUS-005']
Is unique?     False


### 2.2 — Row access by station name

In [68]:
# Use loc to retrieve the full row for station 'BUS-042'
# Now that station_id is the index, this reads like plain English

df.loc["BUS-042"]

district                 Gangseo-gu
latitude                    35.1707
longitude                 129.05573
station_type             background
install_year                   1998
pm25_annual_mean               47.1
no2_annual_mean                42.1
is_active                      True
last_calibration_date    2023-03-15
Name: BUS-042, dtype: object

### 2.3 — `.index =` (positional rename)
> See slide 8 **left panel**. Follow along and observe carefully.

In [69]:
# Take the first 4 rows to demonstrate
subset = df.iloc[:4].copy()
print("Index before rename:", subset.index.tolist())

# Rename all 4 labels in one assignment
# Rule: new list must be EXACTLY the same length as the current index
subset.index = ["Alpha", "Beta", "Gamma", "Delta"]
print("Index after rename: ", subset.index.tolist())
print()
print("Key observation: 'Alpha' is just the name for row 0.")
print("The DATA values did not move — only the labels changed.")
print("This operation is purely POSITIONAL.")
print()

# What happens if the lengths don't match?
try:
    subset.index = ["X", "Y"]          # 2 labels for 4 rows
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Index before rename: ['BUS-001', 'BUS-002', 'BUS-003', 'BUS-004']
Index after rename:  ['Alpha', 'Beta', 'Gamma', 'Delta']

Key observation: 'Alpha' is just the name for row 0.
The DATA values did not move — only the labels changed.
This operation is purely POSITIONAL.

ValueError: Length mismatch: Expected axis has 4 elements, new values have 2 elements


### 2.4 — `.reindex()` (label-aware realignment)
> See slide 8 **right panel**. Follow along and observe carefully.

In [70]:
# Request 4 station IDs — the last one does not exist in the DataFrame
target = ["BUS-003", "BUS-007", "BUS-099", "BUS-GHOST"]

# reindex looks up each label by name in the current index
subset2 = df.reindex(target)

print(subset2[["district", "station_type", "pm25_annual_mean"]])
print()
print("Key observations:")
print("  BUS-003, BUS-007, BUS-099  ->  found in df  ->  data carried across")
print("  BUS-GHOST                  ->  NOT found    ->  entire row is NaN")
print()
# The original df is UNCHANGED — reindex always returns a NEW object
print("Original df.shape is still:", df.shape)

ValueError: cannot reindex on an axis with duplicate labels

- reindex 실행 시 "ValueError: cannot reindex on an axis with duplicate labels" 에러가 발생함.
- 이는 Stage 5에서 중복을 제거하기 전까지 reindex를 사용할 수 없음을 보여주는 예시임.

### 2.5 — Fill in the comparison table

Complete the right column by typing directly in this cell (double-click to edit).

| | `.index = new_labels` | `.reindex(new_labels)` |
|---|---|---|
| **Mechanism** | Positional — row *i* gets `new_labels[i]` | *Label-aware lookup (alignment)* |
| **Length constraint** | Must match exactly — `ValueError` if not | *Any length is OK* |
| **Missing labels** | Raises `ValueError` | *Filled with NaN* |
| **Modifies original?** | Yes, in-place | *No, Returns a new object* |
| **Use when** | Renaming existing labels | *You want to reorder or expand data* |

---
## Stage 3 — Filter with loc

**Goal:** Select only the active roadside stations
using a boolean mask and `loc`.

### 3.1 — Build two boolean Series

In [ ]:
# A boolean mask is a Series of True/False values with the same index as df
# One True per row where the condition holds

mask_active   = df["is_active"] == True          # True where station is active
mask_roadside = df["station_type"] == "roadside"   # True where type is roadside

print("Active stations:   ", mask_active.sum())
print("Roadside stations: ", mask_roadside.sum())
print("Mask length:       ", len(mask_active), "(should equal df.shape[0])")

Active stations:    102
Roadside stations:  48
Mask length:        120 (should equal df.shape[0])


### 3.2 — Combine masks with `&` and apply with `loc`

In [ ]:
# Combine with & (element-wise AND)
# NEVER use Python's 'and' keyword — it doesn't work on Series
combined_mask = mask_active & mask_roadside

roadside_active = df.loc[combined_mask]

print("Active roadside stations:", roadside_active.shape[0])
print("All active?  ", roadside_active["is_active"].all())
print("All roadside?", (roadside_active["station_type"] == "roadside").all())

Active roadside stations: 43
All active?   True
All roadside? True


### 3.3 — Why not `and`? (live error demo)

In [ ]:
# This cell deliberately shows the error you get from using 'and' instead of &
# Run it, read the error, understand why & is required

try:
    wrong_result = df.loc[mask_active and mask_roadside]
except ValueError as e:
    print("ValueError raised:", e)
    print()
    print("Fix: use & instead of 'and'")
    print("     wrap each condition in ( ) when combining")
    print("     e.g.:  df.loc[(df['a'] > 0) & (df['b'] == 'x')]")

ValueError raised: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Fix: use & instead of 'and'
     wrap each condition in ( ) when combining
     e.g.:  df.loc[(df['a'] > 0) & (df['b'] == 'x')]


### 3.4 — Select rows AND columns simultaneously

In [ ]:
# loc takes: df.loc[row_selector, column_selector]
# Select only 4 columns from the filtered rows

cols = ["district", "station_type", "pm25_annual_mean", "is_active"]

roadside_slim = df.loc[combined_mask, cols]

print("Shape:", roadside_slim.shape)        
roadside_slim.head(6)

Shape: (43, 4)


,district,station_type,pm25_annual_mean,is_active
station_id,,,,
BUS-004,Geumjeong-gu,roadside,40.8,True
BUS-005,Nam-gu,roadside,12.4,True
BUS-007,Buk-gu,roadside,43.8,True
BUS-016,Dong-gu,roadside,18.7,True
BUS-019,Haeundae-gu,roadside,46.9,True
BUS-020,Sasang-gu,roadside,37.7,True


### 3.5 — Label slicing (inclusive at both ends)

In [ ]:
# loc slices are INCLUSIVE at the end label — unlike iloc which excludes the end
# Select all stations from BUS-010 through BUS-020

label_slice = df.loc["BUS-010" : "BUS-020"]
print("Rows returned:", label_slice.shape[0])
# Should be 11 rows (BUS-010, 011, 012, ... 020 = 11 stations)

Rows returned: 11


**Stage 3 checkpoint ✓**
- `roadside_active` has fewer rows than `df` — only active roadside stations
- `roadside_slim` has exactly 4 columns
- `label_slice` returns 11 rows

---
## Stage 4 — Add a Computed Column

**Goal:** Add `aqi_ratio = pm25_annual_mean / no2_annual_mean`.
Observe how NaN propagates automatically — no special handling needed.

### 4.1 — Assign the new column

In [ ]:
# Create 'aqi_ratio' as a new column on df
# Syntax: df['new_column'] = expression
# The expression is evaluated row-by-row, vectorially — no loop needed

df["aqi_ratio"] = df["pm25_annual_mean"] / df["no2_annual_mean"]

print("Shape after adding column:", df.shape)   # should be (120, 10)
print("Last column:", df.columns[-1])

Shape after adding column: (120, 10)
Last column: aqi_ratio


### 4.2 — Check NaN propagation

In [ ]:
# If pm25 OR no2 is NaN for a row, the ratio for that row is NaN
# pandas handles this automatically — no if/else needed

null_ratio  = df["aqi_ratio"].isnull().sum()    # fill in: count NaN values
valid_ratio = df["aqi_ratio"].notnull().sum()    # fill in: count non-NaN values

print(f"Valid ratios: {valid_ratio}")
print(f"NaN ratios:   {null_ratio}")
print()
print("Why are there more NaN ratios than NaN values in either source column?")
print("(Think: what happens when you divide NaN by a valid number?)")

Valid ratios: 102
NaN ratios:   18

Why are there more NaN ratios than NaN values in either source column?
(Think: what happens when you divide NaN by a valid number?)


### 4.3 — Describe and explore

In [ ]:
# Summary statistics for the new column
print(df["aqi_ratio"].describe().round(3))

In [ ]:
# Which district has the highest mean aqi_ratio?
# Pattern: groupby -> mean -> sort_values -> head
district_ratio = (
    df.groupby("district")["aqi_ratio"]
      .mean()
      .sort_values(ascending=False)
)

print("Top 5 districts by mean aqi_ratio:")
print(district_ratio.head())
print()
print("District with highest ratio:", district_ratio.idxmax())

Top 5 districts by mean aqi_ratio:
district
Jung-gu      2.191420
Dong-gu      2.155554
Buk-gu       2.014257
Saha-gu      1.923418
Sasang-gu    1.766616
Name: aqi_ratio, dtype: float64

District with highest ratio: Jung-gu


---
## Stage 5 — Find & Drop Duplicates

**Goal:** Detect the two stations with duplicate IDs, remove them,
and verify the fix with `assert` statements.
Rule: *"a cleaning step isn't done until you prove the problem is gone."*

### 5.1 — Mark all rows involved in duplicate IDs

In [ ]:
# keep=False marks ALL copies of a duplicate — not just the second one
# This lets you see the full picture before deciding what to drop

dup_mask = df.index.duplicated(keep=False)

print(f"Rows involved in duplicate IDs: {dup_mask.sum()}")
print()
df[dup_mask][["district", "station_type", "pm25_annual_mean"]]

Rows involved in duplicate IDs: 4



,district,station_type,pm25_annual_mean
station_id,,,
BUS-047,Haeundae-gu,background,26.2
BUS-047,Haeundae-gu,background,21.6
BUS-083,Gijang-gun,roadside,46.3
BUS-083,Gijang-gun,background,15.8


### 5.2 — Find the specific IDs that are duplicated

In [ ]:
# Extract the duplicated station IDs as a plain Python list
dup_ids = df.index[df.index.duplicated()].unique().tolist()
print("Duplicated station IDs:", dup_ids)

Duplicated station IDs: ['BUS-047', 'BUS-083']


### 5.3 — Inspect the duplicate pairs side by side

In [ ]:
# Print both rows for each duplicated ID
# Are they identical, or do they have different values?
for sid in dup_ids:
    print(f"=== Both rows for {sid} ===")
    print(df.loc[sid, ["district","station_type","pm25_annual_mean","no2_annual_mean"]])
    print()

=== Both rows for BUS-047 ===
               district station_type  pm25_annual_mean  no2_annual_mean
station_id                                                             
BUS-047     Haeundae-gu   background              26.2             39.0
BUS-047     Haeundae-gu   background              21.6              NaN

=== Both rows for BUS-083 ===
              district station_type  pm25_annual_mean  no2_annual_mean
station_id                                                            
BUS-083     Gijang-gun     roadside              46.3             27.5
BUS-083     Gijang-gun   background              15.8             15.0



### 5.4 — Drop duplicates, keep first occurrence

In [ ]:
# ~dup_mask flips True/False — keeps rows where the mask is False (= not a duplicate)
# Equivalently: df.index.duplicated(keep='first') marks seconds as True, then ~ keeps firsts

df_clean = df[~df.index.duplicated(keep='first')]

print("Before:", df.shape)
print("After: ", df_clean.shape)
print("Removed:", df.shape[0] - df_clean.shape[0])

Before: (120, 9)
After:  (118, 9)
Removed: 2


### 5.5 — Verify with assert statements

In [ ]:
# Fill in the expected values (___) before running
# If any assertion fails, you get an immediate AssertionError — fix the issue, then re-run

assert df_clean.index.is_unique,           "FAIL: duplicate IDs still present!"
assert df_clean.shape[0] == 118,           "FAIL: unexpected row count!"
assert df_clean.shape[1] == 10,           "FAIL: unexpected column count!"
assert "aqi_ratio" in df_clean.columns,    "FAIL: aqi_ratio column missing!"

print("All assertions passed ✓")
print(f"Clean dataset: {df_clean.shape}")
print(f"Index is unique: {df_clean.index.is_unique}")

All assertions passed ✓
Clean dataset: (118, 10)
Index is unique: True


In [ ]:
# Final null check — NaN counts should be the same as Stage 1
# (We have not imputed anything yet — that is Week 10's topic)
print("Null counts in df_clean:")
print(df_clean.isnull().sum())

Null counts in df_clean:
district                  0
latitude                  0
longitude                 0
station_type              0
install_year              0
pm25_annual_mean         10
no2_annual_mean           8
is_active                 0
last_calibration_date     0
aqi_ratio                17
dtype: int64


In [ ]:
# Final preview of the clean dataset
df_clean.head(8)

,district,latitude,longitude,station_type,install_year,pm25_annual_mean,no2_annual_mean,is_active,last_calibration_date,aqi_ratio
station_id,,,,,,,,,,
BUS-001,Nam-gu,35.32548,128.99372,background,2011,44.4,30.4,True,2023-03-15,1.460526
BUS-002,Seo-gu,35.25493,129.17816,background,2022,28.6,32.7,True,2023-11-30,0.874618
BUS-003,Seo-gu,35.30958,129.29169,background,2013,48.4,13.0,True,2023-03-15,3.723077
BUS-004,Geumjeong-gu,35.21178,129.13685,roadside,2020,40.8,24.6,True,2023-01-20,1.658537
BUS-005,Nam-gu,35.24785,128.90289,roadside,2006,12.4,11.5,True,2023-08-10,1.078261
BUS-006,Nam-gu,35.12374,129.13885,background,2009,53.9,27.3,True,2022-12-01,1.974359
BUS-007,Buk-gu,35.28456,128.89779,roadside,2008,43.8,21.5,True,2023-08-10,2.037209
BUS-008,Geumjeong-gu,35.09287,129.25134,industrial,2020,44.9,15.1,True,2023-08-10,2.973510


**Stage 5 checkpoint ✓**
- `df_clean.shape` == `(118, 10)`
- `df_clean.index.is_unique` == `True`
- All four `assert` statements pass without error
- NaN counts in `pm25_annual_mean` and `no2_annual_mean` are unchanged

---
## Pipeline Recap

Fill in the table from memory, then check against slide 13.

| Stage | What we did | Key methods used |
|-------|-------------|------------------|
| 1 — Load & Inspect | Loaded the CSV and checked initial shape and missing values. | pd.read_csv(), .info(), .head() |
| 2 — Set Index | Set station_id as index and practiced label-based access. | .set_index(), .loc[], .reindex() |
| 3 — Filter (loc) | Extracted 'active' and 'roadside' stations using boolean masks. | df.loc[mask], & operator |
| 4 — Computed Column | Created aqi_ratio and observed NaN propagation during division. | df["new_col"] = ..., .describe() |
| 5 — Deduplicate | Identified duplicate IDs and kept first occurrences to clean data. | Identified duplicate IDs and kept first occurrences to clean data. |

---
## Extension Tasks

Use `df_clean` throughout both tasks.

---

### Task 1 — District-level PM2.5 Summary

1. Compute the **mean `pm25_annual_mean`** for every district.
2. Which district has the **highest** average PM2.5?
3. Use `loc` to extract **all stations in that district** and display their station ID and PM2.5 value, sorted worst-first.

> Hint: `groupby` → `mean()` → `idxmax()`

In [ ]:
# Task 1 — your code here

# Step 1: mean PM2.5 per district
district_pm25 = df_clean.groupby("district")["pm25_annual_mean"].mean(). sort_values(ascending=False)
print("Top 5 districts by mean PM2.5:")
print(district_pm25.head())

# Step 2: worst district
worst_district = district_pm25.idxmax()
print(f"Worst district: {worst_district}")

# Step 3: all stations in that district
worst_stations = df_clean.loc[
    df_clean["district"] == worst_district,
    ["district", "station_type", "pm25_annual_mean"]
].sort_values("pm25_annual_mean", ascending=False)

print(f"All stations in {worst_district}:")
print(worst_stations)

Top 5 districts by mean PM2.5:
district
Dong-gu       53.728571
Buk-gu        39.666667
Jung-gu       38.625000
Saha-gu       38.062500
Gangseo-gu    37.800000
Name: pm25_annual_mean, dtype: float64
Worst district: Dong-gu
All stations in Dong-gu:
           district station_type  pm25_annual_mean
station_id                                        
BUS-099     Dong-gu     roadside             119.7
BUS-039     Dong-gu   industrial              93.3
BUS-110     Dong-gu   background              50.4
BUS-066     Dong-gu   industrial              34.0
BUS-090     Dong-gu     roadside              31.5
BUS-041     Dong-gu   industrial              28.5
BUS-016     Dong-gu     roadside              18.7
BUS-040     Dong-gu   background               NaN


### Task 2 — Legacy vs Modern Stations

1. Add a boolean column `legacy`: `True` for stations installed **before 2010**.
2. Compare the **mean and std of `pm25_annual_mean`** between legacy and modern stations using `groupby`.
3. Write a one-sentence interpretation as a comment in the code cell.

> Hint: boolean comparison returns a Series of `True`/`False` directly

In [ ]:
# Task 2 — your code here

# Step 1: add the 'legacy' boolean column
df_clean["legacy"] = df_clean["install_year"] < 2010

print("Legacy (pre-2010):", df_clean["legacy"].sum(), "stations")
print("Modern (2010+):   ", (~df_clean["legacy"]).sum(), "stations")
print()

# Step 2: compare mean PM2.5
comparison = (
    df_clean.groupby("legacy")["pm25_annual_mean"]
    .agg(["mean", "std", "count"])
    .round(2)
)
comparison.index = ["Modern (2010+)", "Legacy (pre-2010)"]
print("PM2.5 comparison:")
print(comparison)

# Step 3: write your interpretation here
# Observation: legacy stations are ___ polluted than modern ones because ___

Legacy (pre-2010): 61 stations
Modern (2010+):    57 stations

PM2.5 comparison:
                    mean    std  count
Modern (2010+)     33.91  15.41     51
Legacy (pre-2010)  33.08  19.13     57


---
## Reflection

Answer in this cell (double-click to edit). No code required.

**1. `loc` vs `iloc`:**
When would you choose `iloc` over `loc`? Give one concrete situation for each.

*Your answer:* Choose loc when you need to select data based on meaningful labels (e.g., retrieving a specific station by its ID like 'BUS-010'). Choose iloc when you need to select data based on its physical position (e.g., picking the first 10 rows of a dataset regardless of what their IDs are).

---

**2. `.index=` vs `.reindex()`:**
Write the key difference in one sentence without looking at the slides.

*Your answer:* .index= is a positional rename that simply replaces label names without moving data, whereas .reindex() is a label-aware lookup that aligns existing data to a new set of labels.

---

**3. NaN propagation:**
Why is automatic NaN propagation in computed columns a *feature*, not a bug?

*Your answer:* It is a feature because it maintains data integrity by preventing the creation of "hallucinated" or misleading values; it explicitly signals that a result is unknown because the input data was incomplete.

---

**4. The `assert` pattern:**
Why is it better to use `assert` after each cleaning step rather than just printing the shape?

*Your answer:* Unlike print, which requires manual inspection and is easily ignored, assert acts as a hard gate that automatically halts execution if data quality requirements aren't met, preventing errors from cascading through the rest of the pipeline.

---
## Appendix — Dataset Generator
> Run **only** if you do not have `busan_aq_stations.csv` from the course portal.

In [ ]:
# ── Dataset generator ──────────────────────────────────────────────────────────
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
N   = 120

DISTRICTS = [
    "Haeundae-gu","Jung-gu","Nam-gu","Dong-gu","Buk-gu","Saha-gu","Sasang-gu",
    "Yeonje-gu","Suyeong-gu","Geumjeong-gu","Gangseo-gu","Busanjin-gu",
    "Dongnae-gu","Gijang-gun","Seo-gu","Youngdo-gu"
]
TYPES    = ["roadside","background","industrial"]
CAL_DATS = ["2023-03-15","2023-06-01","15/09/2023","2023-11-30","2022-12-01","2023-01-20","2023-08-10"]

ids = [f"BUS-{i:03d}" for i in range(1, N+1)]
ids[47] = "BUS-047"; ids[83] = "BUS-083"          # inject two duplicates

pm25 = rng.uniform(8, 55, N).round(1)
no2  = rng.uniform(10, 45, N).round(1)
pm25[rng.choice(N, 10, replace=False)] = np.nan
no2[rng.choice(N, 9,  replace=False)]  = np.nan
pm25[rng.choice(N, 3,  replace=False)] = rng.uniform(90, 140, 3).round(1)

pd.DataFrame({
    "station_id":            ids,
    "district":              rng.choice(DISTRICTS, N),
    "latitude":              rng.uniform(35.02, 35.38, N).round(5),
    "longitude":             rng.uniform(128.88, 129.30, N).round(5),
    "station_type":          rng.choice(TYPES, N, p=[0.45, 0.35, 0.20]),
    "install_year":          rng.integers(1998, 2023, N),
    "pm25_annual_mean":      pm25,
    "no2_annual_mean":       no2,
    "is_active":             rng.choice([True, False], N, p=[0.85, 0.15]),
    "last_calibration_date": rng.choice(CAL_DATS, N),
}).to_csv("busan_aq_stations.csv", index=False)

print("Generated: busan_aq_stations.csv — return to Stage 1 and run from the top.")